# Multi-Qubit Systems & Entanglement — Live Demo Notebook

**Companion notebook to the talk by Fatima Bint-e-Munir**
Qiskit Fall Fest 2026 · University of Engineering & Technology Lahore, Narowal Campus

Same format as the Talk 2 companion notebook: every section follows the slides in order, and every
"Predict Before You Peek" moment from the deck has a matching cell here.

- **Part 1** — Multi-qubit state representation
- **Part 2** — Tensor products
- **Part 3** — Entanglement (+ hands-on lab)
- **Part 4** — Bell states
- **Part 5** — GHZ states

 **Tip:** Run cells top to bottom — later sections reuse helper functions defined earlier

⚙️ **Qiskit version:** targets **Qiskit 2.x**, 

In [ ]:
!pip install "qiskit>=2.0,<3.0" pylatexenc -q

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram
import numpy as np
import qiskit
import itertools

#%matplotlib inline
np.set_printoptions(precision=3, suppress=True)

print("Qiskit version:", qiskit.__version__)   # should read 2.x

## Part 1 — Multi-Qubit State Representation

*How do we describe two, three, or a hundred qubits at once?*


### From One Qubit to Two

Two independent binary choices always combine into four joint possibilities — same pattern as two
coins, new symbols. Two qubits, labeled q1 and q0, have four joint basis states:
`|00>, |01>, |10>, |11>`.


In [ ]:
# 2 qubits -> 4 basis states. n qubits -> 2**n basis states.
n = 2
basis_labels = ["".join(bits) for bits in itertools.product("01", repeat=n)]
print(f"{n} qubits -> {2**n} basis states:", basis_labels)

### Two Qubits, One Joint System

Unlike two coins, a qubit pair can be spread across all four bitstring outcomes *at once*, each
with its own amplitude:

$$|\psi\rangle = \alpha_{00}|00\rangle + \alpha_{01}|01\rangle + \alpha_{10}|10\rangle + \alpha_{11}|11\rangle$$

This is the first genuinely new thing multi-qubit systems bring: superposition over **joint**
outcomes, not just single-qubit ones.


### The Two-Qubit State Vector

A general two-qubit state is a normalized column vector with one amplitude per basis state.
Normalization still applies: `|a00|^2 + |a01|^2 + |a10|^2 + |a11|^2 = 1`.


In [ ]:
# Example from the slide: (1/2)|00> - (1/2)|01> + (i/2)|10> + (1/2)|11>
psi = Statevector([0.5, -0.5, 0.5j, 0.5])
print(psi)
print("Is normalized?", psi.is_valid())
print("Sum of |amplitude|^2:", sum(abs(a)**2 for a in psi.data))

### In Qiskit — Building & Reading Multi-Qubit States

**Predict Before You Peek:** that statevector prints with `dims=(2, 2)`. Which entry is the
amplitude for `|10>` — the 2nd or the 3rd (index 2)?


In [ ]:
psi = Statevector([0.5, -0.5, 0.5j, 0.5])
print(psi)
print("dims:", psi.dims())
print()

# index k in binary tells you each qubit's value -- qubit 0 on the right (little-endian)
for k, amp in enumerate(psi.data):
    q1, q0 = (k >> 1) & 1, k & 1
    print(f"index {k}  ->  |{q1}{q0}>   amplitude = {amp}")
# index 2 = |10> -- the THIRD entry (0-indexed as index 2), amplitude 0.5j

## Part 2 — Tensor Products

*The one operation that combines separate quantum systems into one.*


 **Predict Before You Peek — Expand a Tensor Product**

`|1> ⊗ |-> = |1> ⊗ (1/√2 |0> - 1/√2 |1>) = ?`

Distribute `|1>` over the sum, then attach it to each term. Run the cell below to check your
answer.


In [ ]:
one = Statevector.from_label("1")
minus = Statevector.from_label("-")

result = one.tensor(minus)
print(result)
print(result.probabilities_dict())
# answer: 1/sqrt(2) |10> - 1/sqrt(2) |11>

### Meet CNOT — A Two-Qubit Gate

Every gate so far has acted on one qubit at a time. **CNOT** is different — it reads one qubit (the
**control**) to decide what to do to the other (the **target**): flip the target, but only when the
control is `|1>`.

| Input | Output | Why |
|---|---|---|
| `\|00>` | `\|00>` | control 0, no flip |
| `\|01>` | `\|11>` | control 1, target flips |
| `\|10>` | `\|10>` | control 0, no flip |
| `\|11>` | `\|01>` | control 1, target flips |

(control = q0, target = q1 — `qc.cx(0, 1)`)


In [ ]:
CX = np.array([[1, 0, 0, 0],
               [0, 0, 0, 1],
               [0, 0, 1, 0],
               [0, 1, 0, 0]])

qc = QuantumCircuit(2)
qc.cx(0, 1)   # control=q0, target=q1
display(qc.draw(output="mpl"))

print("Matches Qiskit's CX matrix?", np.allclose(CX, Operator(qc).data))

## Part 3 — Entanglement

*Correlations between qubits that no classical explanation can account for.*

**A correlation, not a mystery.** Two matched coins in sealed boxes look similar — open one and
you instantly know the other. But each coin had a definite value the whole time; you just didn't
know it. Entangled qubits show the same instant correlation, but genuinely do **not** have a hidden
predetermined value — confirmed by decades of Bell-test experiments (2022 Nobel Prize in Physics).


### The Definition

A joint state is entangled precisely when it is **not** a product state.

| Separable (product state) | Entangled (not separable) |
|---|---|
| `\|phi> ⊗ \|psi>` | `1/√2 \|00> + 1/√2 \|11>` |
| Qubit 1 is definitely `\|phi>`, qubit 2 is definitely `\|psi>` — describable independently | Neither qubit has its own definite state; measuring one instantly tells you the other |

For quantum state vectors, entanglement and correlation are the same thing: *not a product state ⇒
entangled.*


### From Theory to Circuit — Creating Entanglement: H + CNOT

Two gates are all it takes: a Hadamard to create superposition, then a CNOT to correlate it with a
second qubit.

`|0>⊗|0>  -H(q0)->  |+>⊗|0>  -CX(q0->q1)->  1/√2|00> + 1/√2|11>`

**Why the CNOT entangles:** because the control is in superposition, the "flip" and "no-flip"
branches of the CNOT both happen at once, in step with the control — that's exactly what ties the
two qubits together.


In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
display(qc.draw(output="mpl"))

sv = Statevector(qc)
print(sv)
print(sv.probabilities_dict())

 **Hands-On — Build and Measure Your Own Entangled Pair**

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)
counts = job.result()[0].data.meas.get_counts()

print(counts)
plot_histogram(counts)
# only "00" and "11" ever appear -- "01" and "10" never do.
# That's the experimental signature of entanglement: perfectly correlated, but each
# individual outcome is still genuinely random.

## Part 4 — Bell States

*The four canonical entangled pairs, named for physicist John Stewart Bell.*


### The Bell Basis

Every one of these is maximally entangled — and together they form a complete basis for two
qubits: any two-qubit state at all can be written as a combination of these four.

$$|\Phi^+\rangle = \tfrac{1}{\sqrt2}(|00\rangle+|11\rangle) \qquad
|\Phi^-\rangle = \tfrac{1}{\sqrt2}(|00\rangle-|11\rangle)$$
$$|\Psi^+\rangle = \tfrac{1}{\sqrt2}(|01\rangle+|10\rangle) \qquad
|\Psi^-\rangle = \tfrac{1}{\sqrt2}(|01\rangle-|10\rangle)$$


### The Recipe — Building |Φ⁺⟩, Step by Step

1. **Start in `|00>`** — both qubits initialize to `|0>` by default in Qiskit.
2. **Hadamard on q0** — puts the control qubit into `|+>`, an equal superposition.
3. **CNOT, q0 → q1** — ties q1 to whatever q0 becomes. Entanglement complete.


In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
display(qc.draw(output="mpl"))

print(Statevector(qc))   # this is |Phi+>

**From Theory to Data — Measuring |Φ⁺⟩**

1,000 shots of `|Phi+>`, measured and tallied — only two outcomes ever appear.


In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)
counts = job.result()[0].data.meas.get_counts()

print(counts)
plot_histogram(counts)
# perfectly correlated: every shot lands on 00 or 11, roughly 50/50, but always matching

 **Predict Before You Peek — From Φ⁺ to Ψ⁺**

What single one-qubit gate, applied to q0 right after the CNOT, turns `|Phi+>` into `|Psi+>`?
Think in terms of Part 2: `(I ⊗ X)|Phi+>` flips only the q0 half of each term.


In [ ]:
# Phi+ -> Psi+ via X on q0
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(0)
psi_plus = Statevector(qc)
print("Phi+ --X(q0)--> ", psi_plus)
print(psi_plus.probabilities_dict())   # matches |Psi+> = 1/sqrt(2)(|01> + |10>)

# The other two Bell states follow the same idea: apply Z instead of X for a minus sign
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.z(0)
phi_minus = Statevector(qc)
print("\nPhi+ --Z(q0)--> ", phi_minus)   # |Phi->

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(0)
qc.z(0)
psi_minus = Statevector(qc)
print("Phi+ --X then Z (q0)--> ", psi_minus)   # |Psi->
# all four canonical Bell states, built from the one Phi+ recipe plus a single-qubit gate

### Why Bell States Matter

- **Bell tests** — measuring Bell states in different bases produces correlations too strong for
  any classical, local-hidden-variable theory to explain.
- **Quantum teleportation** — a shared Bell pair lets you transmit an unknown qubit's state using
  only classical communication plus entanglement.
- **Superdense coding** — a single qubit, if it's half of a Bell pair, can carry two classical bits
  of information instead of one.


** Quantum Teleportation, Verified**

Alice wants to send Bob her qubit's exact (unknown) state `|psi>`, without moving the qubit and
without ever copying it. The protocol:

1. Alice and Bob share a Bell pair in advance — one qubit each.
2. Alice entangles her message qubit with her half of the pair, then measures both — yielding bits
   `m0, m1`.
3. Alice sends those 2 classical bits to Bob.
4. Bob applies the matching correction — his qubit becomes exactly `|psi>`.

| m0 m1 | Bob applies |
|---|---|
| 00 | I |
| 01 | X |
| 10 | Z |
| 11 | X then Z |

Rather than simulate real mid-circuit measurement (which needs Qiskit's dynamic-circuits API — a
topic of its own), let's verify the protocol algebraically: build the state right up to the point
Alice would measure, then check *all four* possible outcomes and confirm every single correction
reconstructs `|psi>` exactly.


In [ ]:
# An arbitrary "unknown" state Alice wants to send Bob (any normalized 2-vector works)
psi = Statevector([0.6, 0.8j])
print("The state Alice wants to send Bob:", psi)

X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])
I2 = np.eye(2)
correction = {
    (0, 0): I2,
    (0, 1): X,
    (1, 0): Z,
    (1, 1): Z @ X,   # "X then Z" -> apply X first, then Z
}

qc = QuantumCircuit(3)
qc.initialize(psi.data, 0)   # q0 = |psi>, Alice's message qubit
qc.h(1)                       # start the Bell pair
qc.cx(1, 2)                   # q1 (Alice) and q2 (Bob) now share a Bell pair
qc.cx(0, 1)                   # Alice's Bell-basis measurement, step 1
qc.h(0)                       # Alice's Bell-basis measurement, step 2

full_state = Statevector(qc)   # the state right before Alice measures q0 and q1

print(f"\n{'m0':>3} {'m1':>3} | {'P(this outcome)':>16} | fidelity of Bob's corrected qubit to |psi>")
print("-" * 62)
for m0, m1 in itertools.product([0, 1], repeat=2):
    idx0 = m1 * 2 + m0          # index where q2 (Bob) = 0
    idx1 = 4 + m1 * 2 + m0      # index where q2 (Bob) = 1
    bob_amps = np.array([full_state.data[idx0], full_state.data[idx1]])
    prob = float(np.sum(np.abs(bob_amps) ** 2))
    bob_state = bob_amps / np.linalg.norm(bob_amps)

    corrected = correction[(m0, m1)] @ bob_state
    fidelity = abs(np.vdot(psi.data, corrected)) ** 2   # 1.0 means an exact match (up to phase)

    print(f"{m0:>3} {m1:>3} | {prob:>16.3f} | {fidelity:.6f}")
# every outcome gives fidelity 1.0 -- Bob's corrected qubit is *exactly* |psi>, every single time

### Closing the Loop — Why This Isn't Magic

- **Nothing outruns light** — only 2 classical bits ever travel from Alice to Bob.
- **Nothing is cloned** — the instant Alice measures, her original qubit collapses; there is only
  ever ONE copy, exactly as No-Cloning requires.
- **Nothing disappears and reappears** — Bob's physical qubit was there the whole time; only its
  *state* changes, via two ordinary corrective gates.


## Part 5 — GHZ States

*Entanglement stretched across three qubits or more, named for Greenberger, Horne & Zeilinger.*


### From Two Qubits to Three

Nothing about entanglement stops at two qubits — the same idea extends to any number:

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}|000\rangle + \frac{1}{\sqrt{2}}|111\rangle$$

Just like the Bell state, this is not a product state — you cannot describe any one qubit on its
own. Measure any single qubit and you get 0 or 1 with 50/50 odds, but the moment you do, the other
two collapse to match it exactly.


### The Recipe, Extended

Same pattern as the Bell state — one Hadamard, then a CNOT fanning out from that same control
qubit to every other qubit:

```
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)
```

q0 does all the work: the Hadamard puts it in superposition, and each CNOT copies that
superposition onto one more qubit. Add a third CX for a fourth qubit, and so on — the pattern
scales to any n.


In [ ]:
def make_ghz(n):
    """Build an n-qubit GHZ state: one H, then n-1 CNOTs fanning out from qubit 0."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for target in range(1, n):
        qc.cx(0, target)
    return qc

qc = make_ghz(3)
display(qc.draw(output="mpl"))

sv = Statevector(qc)
print(sv)
print(sv.probabilities_dict())

# Not a product state either -- check with the same tool from Part 2
rank, s = schmidt_rank_qubit0(sv)
print("\nSchmidt rank (qubit 0 vs. the rest):", rank, " -- genuinely entangled")

In [ ]:
# The pattern generalizes: only all-0 and all-1 bitstrings ever appear, for any n
for n in [2, 3, 4, 5]:
    sv = Statevector(make_ghz(n))
    probs = sv.probabilities_dict()
    print(f"n={n}: {probs}")

### Compare & Contrast — GHZ vs. Bell

| Bell state (2 qubits) | GHZ state (n qubits) |
|---|---|
| `1/√2\|00> + 1/√2\|11>` | `1/√2\|00...0> + 1/√2\|11...1>` |
| One CNOT, one pair entangled | n-1 CNOTs, all fanned from one control |
| 4 basis states, 2 appear | 2ⁿ basis states, only 2 ever appear |
| Losing one qubit destroys the pair entirely | Losing just one qubit still leaves the rest correlated as a smaller GHZ-like state |


 **From Theory to Data — Measuring a GHZ State**

Out of 8 possible 3-bit outcomes, only `000` and `111` ever appear.


In [ ]:
qc = make_ghz(3)
qc.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)
counts = job.result()[0].data.meas.get_counts()

print(counts)
plot_histogram(counts)
# only 000 and 111 ever occur -- all three qubits agree, every single time

 **Predict Before You Peek — Measuring Just One Qubit**

`|GHZ> = 1/√2|000> + 1/√2|111>` — measure only q0 — what happens to q1 and q2? Each outcome has
probability 1/2. Getting 0 should collapse everything to `|000>`; getting 1 should collapse
everything to `|111>` — run the cell a few times to see both happen.


In [ ]:
sv = Statevector(make_ghz(3))
outcome, collapsed = sv.measure([0])   # measure only qubit 0

print("Measured q0, got:", outcome)
print("The other two qubits collapsed along with it:", collapsed)
print(collapsed.probabilities_dict())
# the other two qubits were never independent to begin with -- they immediately
# match whatever q0 turned out to be, with certainty

In [1]:
**Fatima Bint-e-Munir, Qiskit Fall Fest 2026, University of Engineering and Technology Lahore, Narowal Campus**


SyntaxError: invalid syntax (4033916511.py, line 1)